# ⚡ Smart Power — Transformation

**Question:** When is electricity in the Netherlands cheapest **and** cleanest, and how much is driven by renewable generation?

The raw data (90 days, hourly, all in UTC) is pulled by `src/ingest.py`. This notebook
**loads that raw data**, transforms it, computes the renewable share, joins everything into
one clean table, and stores it in a SQLite database.

## 1. Load raw data (produced by `src/ingest.py`)

In [1]:
import pandas as pd

# Load the raw CSVs
weather = pd.read_csv("../data/raw_weather.csv")
prices  = pd.read_csv("../data/raw_prices.csv")
gen     = pd.read_csv("../data/raw_generation.csv")

# Make sure every timestamp is a proper UTC datetime
for name, df in [("weather", weather), ("prices", prices), ("generation", gen)]:
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    print(name, df.shape)

weather (2184, 3)
prices (2184, 2)
generation (8640, 11)


## 2. Transform generation → hourly + renewable share

Generation is 15-min, so we resample it to hourly, then compute the **renewable share** for every hour — the key variable of the project.

In [2]:
# Resample generation from 15-min to hourly (average power per hour)
gen = gen.set_index("timestamp")
gen_hourly = gen.resample("h").mean()

print(gen_hourly.shape)   # ~2160 rows = 90 days x 24 hours
gen_hourly.head()

(2160, 10)


,Biomass,Fossil Gas,Fossil Hard coal,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Waste,Wind Offshore,Wind Onshore
timestamp,,,,,,,,,,
2026-05-05 22:00:00+00:00,19.96550,6302.35675,2278.71450,0.0,0.0,1997.01675,0.0,261.75600,1974.55050,818.53700
2026-05-05 23:00:00+00:00,22.50475,4903.80900,2271.07600,0.0,0.0,1898.88175,0.0,263.13375,1866.97375,859.98575
2026-05-06 00:00:00+00:00,26.25700,4376.67800,2237.65100,0.0,0.0,1887.43550,0.0,262.67700,1945.22975,860.59675
2026-05-06 01:00:00+00:00,26.30975,4146.75825,2273.05400,0.0,0.0,1826.83950,0.0,262.98300,2088.13700,862.65500
2026-05-06 02:00:00+00:00,26.35425,4106.26875,2279.21425,0.0,0.0,1878.63500,0.0,266.13725,2199.99125,865.47475


In [3]:
# Compute the renewable share for every hour
production_cols = gen_hourly.columns.tolist()          # all generation types

# Pick renewable columns by name (Wind Offshore, Wind Onshore, Solar, Hydro..., Biomass)
renewable_keywords = ["Solar", "Wind", "Hydro", "Biomass"]
renewable_cols = [c for c in production_cols
                  if any(k in c for k in renewable_keywords)]
print("Renewable columns:", renewable_cols)

gen_hourly["total_generation"]     = gen_hourly[production_cols].sum(axis=1)
gen_hourly["renewable_generation"] = gen_hourly[renewable_cols].sum(axis=1)
gen_hourly["renewable_share"]      = (
    gen_hourly["renewable_generation"] / gen_hourly["total_generation"]
)

gen_hourly[["total_generation", "renewable_generation", "renewable_share"]].head()

Renewable columns: ['Biomass', 'Hydro Run-of-river and poundage', 'Solar', 'Wind Offshore', 'Wind Onshore']


,total_generation,renewable_generation,renewable_share
timestamp,,,
2026-05-05 22:00:00+00:00,13652.89700,2813.05300,0.206041
2026-05-05 23:00:00+00:00,12086.36475,2749.46425,0.227485
2026-05-06 00:00:00+00:00,11596.52500,2832.08350,0.244218
2026-05-06 01:00:00+00:00,11486.73650,2977.10175,0.259177
2026-05-06 02:00:00+00:00,11622.07550,3091.82025,0.266030


In [4]:
# Sanity check: renewable_share should sit between 0 and 1
print(gen_hourly["renewable_share"].describe())

count    2160.000000
mean        0.198737
std         0.146982
min         0.000164
25%         0.084932
50%         0.162338
75%         0.279118
max         0.690521
Name: renewable_share, dtype: float64


## 3. Join + store in a database

Merge the three hourly tables on `timestamp` into one clean dataset, then save it to a SQLite database and check it with SQL (Unit 4 revision).

In [5]:
# Bring gen_hourly's index back as a column so we can merge on it
gen_hourly = gen_hourly.reset_index()

# Merge the three tables on timestamp.
# how="inner" keeps only the hours present in ALL three tables (no missing values)
df = prices.merge(gen_hourly, on="timestamp", how="inner")
df = df.merge(weather, on="timestamp", how="inner")

print(df.shape)   # ~2150 rows (the overlapping hours)
df.head()

(2158, 17)


,timestamp,electricity_price,Biomass,Fossil Gas,Fossil Hard coal,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Waste,Wind Offshore,Wind Onshore,total_generation,renewable_generation,renewable_share,wind_speed,solar_radiation
0,2026-05-06 00:00:00+00:00,0.13,26.25700,4376.67800,2237.65100,0.0,0.0,1887.43550,0.00000,262.67700,1945.22975,860.59675,11596.5250,2832.08350,0.244218,NaN,NaN
1,2026-05-06 01:00:00+00:00,0.13,26.30975,4146.75825,2273.05400,0.0,0.0,1826.83950,0.00000,262.98300,2088.13700,862.65500,11486.7365,2977.10175,0.259177,NaN,NaN
2,2026-05-06 02:00:00+00:00,0.13,26.35425,4106.26875,2279.21425,0.0,0.0,1878.63500,0.00000,266.13725,2199.99125,865.47475,11622.0755,3091.82025,0.266030,NaN,NaN
3,2026-05-06 03:00:00+00:00,0.15,46.66050,4337.70700,2274.97050,0.0,0.0,2243.83075,0.00000,257.54950,2260.01175,881.41950,12302.1495,3188.09175,0.259149,NaN,NaN
4,2026-05-06 04:00:00+00:00,0.16,199.04875,4523.28925,2159.12450,0.0,0.0,2781.07850,4.27325,245.96225,2389.63275,854.74475,13157.1540,3447.69950,0.262040,NaN,NaN


In [6]:
# Keep the core columns for analysis + machine learning
clean = df[[
    "timestamp",
    "electricity_price",
    "renewable_share",
    "renewable_generation",
    "total_generation",
    "wind_speed",
    "solar_radiation",
]].copy()

print(clean.shape)
clean.head()

(2158, 7)


,timestamp,electricity_price,renewable_share,renewable_generation,total_generation,wind_speed,solar_radiation
0,2026-05-06 00:00:00+00:00,0.13,0.244218,2832.08350,11596.5250,NaN,NaN
1,2026-05-06 01:00:00+00:00,0.13,0.259177,2977.10175,11486.7365,NaN,NaN
2,2026-05-06 02:00:00+00:00,0.13,0.266030,3091.82025,11622.0755,NaN,NaN
3,2026-05-06 03:00:00+00:00,0.15,0.259149,3188.09175,12302.1495,NaN,NaN
4,2026-05-06 04:00:00+00:00,0.16,0.262040,3447.69950,13157.1540,NaN,NaN


In [7]:
from sqlalchemy import create_engine

# SQLite does not handle timezone-aware datetimes well -> drop the tz (values stay UTC)
clean["timestamp"] = clean["timestamp"].dt.tz_localize(None)

engine = create_engine("sqlite:///../data/smart_power.db")
clean.to_sql("hourly_data", engine, if_exists="replace", index=False)

print("Saved clean table to data/smart_power.db")

Saved clean table to data/smart_power.db


In [8]:
# Read it back with SQL to confirm
print(pd.read_sql("SELECT * FROM hourly_data LIMIT 5", engine))

print(pd.read_sql("""
    SELECT COUNT(*)                          AS rows,
           ROUND(AVG(electricity_price), 3)  AS avg_price,
           ROUND(AVG(renewable_share), 3)    AS avg_renewable
    FROM hourly_data
""", engine))

                    timestamp  electricity_price  renewable_share  \
0  2026-05-06 00:00:00.000000               0.13         0.244218   
1  2026-05-06 01:00:00.000000               0.13         0.259177   
2  2026-05-06 02:00:00.000000               0.13         0.266030   
3  2026-05-06 03:00:00.000000               0.15         0.259149   
4  2026-05-06 04:00:00.000000               0.16         0.262040   

   renewable_generation  total_generation wind_speed solar_radiation  
0            2832.08350        11596.5250       None            None  
1            2977.10175        11486.7365       None            None  
2            3091.82025        11622.0755       None            None  
3            3188.09175        12302.1495       None            None  
4            3447.69950        13157.1540       None            None  
   rows  avg_price  avg_renewable
0  2158      0.127          0.199


In [9]:
print(clean.isna().sum())
print(clean[["wind_speed", "solar_radiation"]].tail())

timestamp                 0
electricity_price         0
renewable_share           0
renewable_generation      0
total_generation          0
wind_speed              528
solar_radiation         528
dtype: int64
      wind_speed  solar_radiation
2153        13.7            371.0
2154        11.9            218.0
2155         5.8             82.0
2156         6.1              5.0
2157         8.6              0.0


In [10]:
clean = clean.dropna().reset_index(drop=True)
print(clean.shape)

# Re-save the clean table without missing rows
clean.to_sql("hourly_data", engine, if_exists="replace", index=False)

(1630, 7)


1630